In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


In [4]:
DATA_DIR = Path(".")
input_file = DATA_DIR / "raw_sales.csv"

df = pd.read_csv(input_file)
print("First five rows:")
display(df.head())
print("Original shape (rows, columns):", df.shape)
print("\\nData types:")
print(df.dtypes)


First five rows:


,Order_ID,Customer,Age,City,Product,Price,Quantity,Payment_Method,Order_Date
0,ORD-1001,Asha Rao,29.0,Hyderabad,Laptop,65000.0,1.0,Credit Card,2026-09-01
1,ORD-1002,Ravi Kumar,34.0,Vijayawada,Mouse,850.0,2.0,UPI,2026-09-02
2,ORD-1003,Meena Devi,NaN,visakhapatnam,Keyboard,1800.0,1.0,upi,2026-09-03
3,ORD-1004,Kiran Reddy,41.0,NaN,Monitor,12500.0,1.0,Debit Card,2026-09-03
4,ORD-1005,Sita Lakshmi,27.0,Guntur,Headphones,NaN,1.0,Cash,2026-09-04


Original shape (rows, columns): (21, 9)
\nData types:
Order_ID           object
Customer           object
Age               float64
City               object
Product            object
Price             float64
Quantity          float64
Payment_Method     object
Order_Date         object
dtype: object


In [5]:
print("Missing values by column:")
display(df.isnull().sum().to_frame("missing_count"))

print("Exact duplicate rows:", df.duplicated().sum())

print("Numerical summary before cleaning:")
display(df.describe())


Missing values by column:


,missing_count
Order_ID,0
Customer,0
Age,3
City,1
Product,0
Price,2
Quantity,1
Payment_Method,1
Order_Date,0


Exact duplicate rows: 1
Numerical summary before cleaning:


,Age,Price,Quantity
count,18.000000,19.000000,20.000000
mean,32.833333,17334.210526,1.550000
std,5.533322,26951.968866,0.825578
min,24.000000,800.000000,1.000000
25%,29.000000,1375.000000,1.000000
50%,32.500000,2500.000000,1.000000
75%,35.750000,13000.000000,2.000000
max,45.000000,72000.000000,4.000000


In [6]:
rows_before_dedup = len(df)
df = df.drop_duplicates().copy()
rows_after_dedup = len(df)

print(f"Rows before duplicate removal: {rows_before_dedup}")
print(f"Exact duplicate rows removed: {rows_before_dedup - rows_after_dedup}")
print(f"Rows after duplicate removal: {rows_after_dedup}")


Rows before duplicate removal: 21
Exact duplicate rows removed: 1
Rows after duplicate removal: 20


In [7]:
# Strip leading/trailing whitespace in text columns first.
text_columns = df.select_dtypes(include=["object", "string"]).columns
for col in text_columns:
    df[col] = df[col].astype("string").str.strip()

# Normalize fields where capitalization should not carry meaning.
# Keep customer/product display names in title case; normalize categories consistently.
df["City"] = df["City"].str.title()
df["Payment_Method"] = df["Payment_Method"].str.lower().str.replace(r"\\s+", " ", regex=True)
df["Payment_Method"] = df["Payment_Method"].replace({
    "credit card": "credit card",
    "debit card": "debit card",
    "upi": "upi",
    "cash": "cash"
})
df["Customer"] = df["Customer"].str.title()
df["Product"] = df["Product"].str.title()

display(df.head())


,Order_ID,Customer,Age,City,Product,Price,Quantity,Payment_Method,Order_Date
0,ORD-1001,Asha Rao,29.0,Hyderabad,Laptop,65000.0,1.0,credit card,2026-09-01
1,ORD-1002,Ravi Kumar,34.0,Vijayawada,Mouse,850.0,2.0,upi,2026-09-02
2,ORD-1003,Meena Devi,NaN,Visakhapatnam,Keyboard,1800.0,1.0,upi,2026-09-03
3,ORD-1004,Kiran Reddy,41.0,<NA>,Monitor,12500.0,1.0,debit card,2026-09-03
4,ORD-1005,Sita Lakshmi,27.0,Guntur,Headphones,NaN,1.0,cash,2026-09-04


In [8]:
# Convert fields that should be numeric. Invalid entries become NaN.
numeric_columns = ["Age", "Price", "Quantity"]
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Median is robust to skewed sales prices and outliers.
for col in numeric_columns:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)

# Categorical columns: use mode (most frequent value) where available.
categorical_columns = ["City", "Payment_Method", "Customer", "Product"]
for col in categorical_columns:
    mode_values = df[col].mode(dropna=True)
    fill_value = mode_values.iloc[0] if not mode_values.empty else "Unknown"
    df[col] = df[col].fillna(fill_value)

# Dates are parsed consistently; any invalid date can be flagged as missing.
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")
print("Missing values after imputation:")
display(df.isnull().sum().to_frame("missing_count"))


Missing values after imputation:


,missing_count
Order_ID,0
Customer,0
Age,0
City,0
Product,0
Price,0
Quantity,0
Payment_Method,0
Order_Date,0


In [9]:
print("Cleaned shape (rows, columns):", df.shape)
print("Remaining exact duplicates:", df.duplicated().sum())
print("Total missing cells:", int(df.isnull().sum().sum()))

assert df.duplicated().sum() == 0, "Duplicate rows remain."
assert int(df.isnull().sum().sum()) == 0, "Missing values remain."

# Business-rule checks
assert (df["Price"] >= 0).all(), "Negative prices found."
assert (df["Quantity"] >= 0).all(), "Negative quantities found."

print("Validation passed.")


Cleaned shape (rows, columns): (20, 9)
Remaining exact duplicates: 0
Total missing cells: 0
Validation passed.


In [10]:
# Save dates in a readable CSV format.
df["Order_Date"] = df["Order_Date"].dt.strftime("%Y-%m-%d")

output_file = DATA_DIR / "cleaned_sales.csv"
df.to_csv(output_file, index=False)

summary = df.describe(include="all").transpose()
summary_file = DATA_DIR / "summary_statistics.csv"
summary.to_csv(summary_file)

print(f"Cleaned dataset saved to: {output_file}")
print(f"Summary statistics saved to: {summary_file}")
display(summary)


Cleaned dataset saved to: cleaned_sales.csv
Summary statistics saved to: summary_statistics.csv


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Order_ID,20,20,ORD-1001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer,20,19,Ravi Kumar,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,20.0,NaN,NaN,NaN,33.05,5.155222,24.0,29.75,33.0,35.25,45.0
City,20,13,Hyderabad,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Product,20,6,Mouse,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Price,20.0,NaN,NaN,NaN,13452.5,24007.704218,800.0,1587.5,2350.0,11975.0,72000.0
Quantity,20.0,NaN,NaN,NaN,1.55,0.825578,1.0,1.0,1.0,2.0,4.0
Payment_Method,20,4,upi,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Order_Date,20,12,2026-09-02,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
raw_df = pd.read_csv(DATA_DIR / "raw_sales.csv")
comparison = pd.DataFrame({
    "Metric": ["Rows", "Columns", "Missing cells", "Exact duplicate rows"],
    "Before cleaning": [
        len(raw_df),
        raw_df.shape[1],
        int(raw_df.isnull().sum().sum()),
        int(raw_df.duplicated().sum())
    ],
    "After cleaning": [
        len(df),
        df.shape[1],
        int(df.isnull().sum().sum()),
        int(df.duplicated().sum())
    ]
})
display(comparison)


,Metric,Before cleaning,After cleaning
0,Rows,21,20
1,Columns,9,9
2,Missing cells,8,0
3,Exact duplicate rows,1,0
